In [1]:
import pandas as pd

# Läs bara de första 10 raderna
df = pd.read_parquet("../data/fhvhv_tripdata_2020-01.parquet")

df = df.head(10)

print(df.shape)
print(df.dtypes)
df.head()




(10, 24)
hvfhs_license_num                  str
dispatching_base_num               str
originating_base_num               str
request_datetime        datetime64[us]
on_scene_datetime       datetime64[us]
pickup_datetime         datetime64[us]
dropoff_datetime        datetime64[us]
PULocationID                     int64
DOLocationID                     int64
trip_miles                     float64
trip_time                        int64
base_passenger_fare            float64
tolls                          float64
bcf                            float64
sales_tax                      float64
congestion_surcharge           float64
airport_fee                    float64
tips                           float64
driver_pay                     float64
shared_request_flag                str
shared_match_flag                  str
access_a_ride_flag                 str
wav_request_flag                   str
wav_match_flag                     str
dtype: object


,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,B02864,B02864,2020-01-01 00:40:13,2020-01-01 00:43:34,2020-01-01 00:45:34,2020-01-01 01:02:20,148,90,1.93,...,2.70,2.75,NaN,0.0,18.25,N,N,,N,N
1,HV0003,B02682,B02682,2020-01-01 00:42:31,2020-01-01 00:46:33,2020-01-01 00:47:50,2020-01-01 00:53:23,114,79,0.81,...,1.31,2.75,NaN,0.0,10.84,N,N,,N,N
2,HV0003,B02764,B02764,2020-01-01 00:01:42,2020-01-01 00:02:06,2020-01-01 00:04:37,2020-01-01 00:21:49,4,125,2.53,...,1.39,2.75,NaN,3.0,11.73,N,N,,N,N
3,HV0003,B02764,B02764,2020-01-01 00:21:23,2020-01-01 00:26:02,2020-01-01 00:26:36,2020-01-01 00:33:00,231,113,1.11,...,0.75,2.75,NaN,0.0,5.84,N,N,,N,N
4,HV0003,B02764,B02764,2020-01-01 00:32:20,2020-01-01 00:37:06,2020-01-01 00:37:49,2020-01-01 00:46:59,114,144,1.10,...,1.03,2.75,NaN,0.0,7.69,N,N,,N,N


In [3]:
import sys
import importlib
import pandas as pd

sys.path.insert(0, "../src")

# --- Kör Dask pipeline ---
import transformDask
importlib.reload(transformDask)
print("=== Kör Dask ===")
dask_timings = transformDask.process_data()

# --- Kör Spark pipeline ---
import transformSpark
importlib.reload(transformSpark)
print("\n=== Kör Spark ===")
spark = transformSpark.create_spark_session()
spark_timings = transformSpark.process_data(spark)
spark.stop()

# --- Jämförelsetabell ---
operations = list(dask_timings.keys())
df_compare = pd.DataFrame({
    "Operation":  operations,
    "Dask (s)":   [round(dask_timings[k], 2) for k in operations],
    "Spark (s)":  [round(spark_timings.get(k, float("nan")), 2) for k in operations],
})
df_compare["Snabbast"] = df_compare.apply(
    lambda r: "Dask" if r["Dask (s)"] < r["Spark (s)"] else "Spark", axis=1
)

print("\n=== Exekveringstid: Dask vs Spark ===")
print(df_compare.to_string(index=False))
print(f"\nDask total:  {round(dask_timings['Total pipeline'], 2)}s")
print(f"Spark total: {round(spark_timings['Total pipeline'], 2)}s")

2026-03-17 10:18:47,767 [TransformationDask] Hittade 34 parquet-filer i c:\Users\Patrik\Documents\GitHub\BigDataPipeline-GroupA4\notebooks\..\data
2026-03-17 10:18:47,801 [TransformationDask] [TIMING] Inläsning: 0.03s
2026-03-17 10:18:47,802 [TransformationDask] Rensar bort ogiltig data (Null-värden)...
2026-03-17 10:18:47,805 [TransformationDask] [TIMING] Filtrering + dropna: 0.00s
2026-03-17 10:18:47,806 [TransformationDask] Joinar med taxizoner...
2026-03-17 10:18:47,830 [TransformationDask] [TIMING] Join med taxizoner: 0.02s
2026-03-17 10:18:47,831 [TransformationDask] Aggregerar data per borough...


=== Kör Dask ===


2026-03-17 10:20:13,179 [TransformationDask] [TIMING] Aggregation per borough: 85.35s
2026-03-17 10:20:13,216 [TransformationDask] Kör window function - rankar zoner per borough...


pickup_borough  antal_resor  snitt_pris
     Manhattan    189111821       23.13
      Brooklyn    135090260       19.68
        Queens     93833765       23.03
         Bronx     66971157       17.41
 Staten Island      6660044       19.15
           EWR         6240       56.90
       Unknown           70       40.74


2026-03-17 10:21:41,718 [TransformationDask] [TIMING] Window function: 88.50s
2026-03-17 10:21:41,790 [TransformationDask] [TIMING] Spara till parquet: 0.05s
2026-03-17 10:21:41,791 [TransformationDask] [TIMING] Total pipeline: 174.02s
2026-03-17 10:21:41,791 [TransformationDask] Pipeline färdig!
2026-03-17 10:21:41,826 [Transformation] Startar Apache Spark-session...


...

+--------------+-----------+----------+
|pickup_borough|antal_resor|snitt_pris|
+--------------+-----------+----------+
|     Manhattan|  189111821|     23.13|
|      Brooklyn|  135090260|     19.68|
|        Queens|   93833765|     23.03|
|         Bronx|   66971157|     17.41|
| Staten Island|    6660044|     19.15|
|           N/A|      30392|     23.85|
|           EWR|       6240|      56.9|
|       Unknown|         70|     40.74|
+--------------+-----------+----------+



2026-03-17 10:22:24,709 [Transformation] [TIMING] Window function: 22.11s


+--------------+--------------------------------+-----------+----+
|pickup_borough|pickup_zone                     |antal_resor|rank|
+--------------+--------------------------------+-----------+----+
|Bronx         |Mott Haven/Port Morris          |3173179    |1   |
|Bronx         |East Concourse/Concourse Village|2787260    |2   |
|Bronx         |Soundview/Castle Hill           |2728380    |3   |
|Brooklyn      |Crown Heights North             |7508783    |1   |
|Brooklyn      |East New York                   |5925478    |2   |
|Brooklyn      |Bushwick South                  |5787060    |3   |
|EWR           |Newark Airport                  |6240       |1   |
|Manhattan     |East Village                    |6945011    |1   |
|Manhattan     |TriBeCa/Civic Center            |5343285    |2   |
|Manhattan     |Central Harlem North            |5068683    |3   |
|N/A           |Outside of NYC                  |30392      |1   |
|Queens        |JFK Airport                     |6398397    |1

2026-03-17 10:23:07,678 [Transformation] [TIMING] Spara till parquet: 42.97s
2026-03-17 10:23:07,679 [Transformation] [TIMING] Total pipeline: 85.00s



=== Exekveringstid: Dask vs Spark ===
              Operation  Dask (s)  Spark (s) Snabbast
              Inläsning      0.03       0.42     Dask
    Filtrering + dropna      0.00       0.02     Dask
     Join med taxizoner      0.02       0.04     Dask
Aggregation per borough     85.35      19.12    Spark
        Window function     88.50      22.11    Spark
     Spara till parquet      0.05      42.97     Dask
         Total pipeline    174.02      85.00    Spark

Dask total:  174.02s
Spark total: 85.0s


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Exkludera "Total pipeline" från stapeldiagrammet
ops = df_compare[df_compare["Operation"] != "Total pipeline"].reset_index(drop=True)
total = df_compare[df_compare["Operation"] == "Total pipeline"].iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Exekveringstid: Dask vs Spark (34 parquet-filer)", fontsize=14, fontweight="bold")

# Vänster: per operation
x = np.arange(len(ops))
w = 0.35
b1 = axes[0].bar(x - w/2, ops["Dask (s)"],  w, label="Dask",  color="#4C9BE8", edgecolor="white")
b2 = axes[0].bar(x + w/2, ops["Spark (s)"], w, label="Spark", color="#F28C38", edgecolor="white")
axes[0].set_xticks(x)
axes[0].set_xticklabels(ops["Operation"], rotation=30, ha="right", fontsize=9)
axes[0].set_ylabel("Sekunder (s)")
axes[0].set_title("Per operation")
axes[0].legend()
axes[0].bar_label(b1, fmt="%.2f", fontsize=8, padding=2)
axes[0].bar_label(b2, fmt="%.2f", fontsize=8, padding=2)

# Höger: total pipeline
vals = [total["Dask (s)"], total["Spark (s)"]]
bars = axes[1].bar(["Dask", "Spark"], vals, color=["#4C9BE8", "#F28C38"], edgecolor="white", width=0.4)
axes[1].bar_label(bars, fmt="%.2fs", fontsize=11, fontweight="bold", padding=4)
axes[1].set_ylabel("Sekunder (s)")
axes[1].set_title("Total pipeline")
axes[1].set_ylim(0, max(vals) * 1.2)

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [5]:
df1 = pd.read_parquet("../data/fhvhv_tripdata_2022-10.parquet")
df = df.head(10)

In [4]:
print(df1.shape)
print(df1.dtypes)
df1.head()


(19306090, 24)
hvfhs_license_num                  str
dispatching_base_num               str
originating_base_num               str
request_datetime        datetime64[us]
on_scene_datetime       datetime64[us]
pickup_datetime         datetime64[us]
dropoff_datetime        datetime64[us]
PULocationID                     int64
DOLocationID                     int64
trip_miles                     float64
trip_time                        int64
base_passenger_fare            float64
tolls                          float64
bcf                            float64
sales_tax                      float64
congestion_surcharge           float64
airport_fee                    float64
tips                           float64
driver_pay                     float64
shared_request_flag                str
shared_match_flag                  str
access_a_ride_flag                 str
wav_request_flag                   str
wav_match_flag                     str
dtype: object


,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,B03404,B03404,2022-10-01 00:07:49,2022-10-01 00:10:31,2022-10-01 00:10:31,2022-10-01 00:21:47,235,174,3.65,...,1.24,0.0,0.0,0.00,12.39,N,N,,N,N
1,HV0003,B03404,B03404,2022-10-01 00:30:24,2022-10-01 00:33:54,2022-10-01 00:35:23,2022-10-01 00:47:21,241,254,2.33,...,1.03,0.0,0.0,0.00,11.23,N,N,,N,N
2,HV0003,B03404,B03404,2022-10-01 00:49:44,2022-10-01 00:52:50,2022-10-01 00:54:01,2022-10-01 01:08:20,174,250,5.69,...,1.59,0.0,0.0,0.00,16.38,N,N,,N,N
3,HV0003,B03404,B03404,2022-10-01 00:47:46,2022-10-01 00:48:48,2022-10-01 00:52:08,2022-10-01 01:14:31,211,265,6.05,...,0.00,0.0,0.0,0.00,32.25,N,N,,N,N
4,HV0003,B03404,B03404,2022-10-01 00:06:17,2022-10-01 00:06:38,2022-10-01 00:07:06,2022-10-01 00:13:54,44,44,1.56,...,0.71,0.0,0.0,0.02,8.26,N,N,,N,N
